In [1]:
# Cell 1: Environment Setup

include("../FullWorkflow/scripts/dictionaries.jl")
include("../FullWorkflow/scripts/helpers.jl")
using .FullWorkflowHelpers
using OMJulia

# --- Configuration ---

# 1. Directory containing the single-file model
MODEL_DIR = abspath("models")

# 2. Select the model to build
MODEL = "MyBESS"

# 3. Path to the selected model file
MODEL_FILE_PATH = joinpath(MODEL_DIR, MODEL * ".mo")

# 4. Path to the Dynawo package.mo
DYNAWO_PKG_PATH   = "/home/clarafercas/Model_library/dynawo/dynawo/sources/Models/Modelica/Dynawo/package.mo"

# 5. Path to the Modelica package.mo
MODELICA_PKG_PATH = "/home/clarafercas/dynawo/OpenModelica/lib/omlibrary/Modelica/package.mo"

# 6. INIT model selection for components with multiple INIT profiles (leave empty for default).
INIT_MODEL_BY_COMPONENT = Dict{String, String}(
    # "generatorSynchronous" => "GeneratorSynchronousInt_INIT",
)

# 7. Slack component (leave empty to disable slack-specific handling).
SLACK_COMPONENT = ""

""

In [2]:
# Cell 2: OpenModelica Setup + Single Model Validation

# 1. Start OMC and load libraries
omc = OMJulia.OMCSession()
omc_call(omc, "loadFile(\"$MODELICA_PKG_PATH\")")
omc_call(omc, "loadModel(Complex)")
omc_call(omc, "loadModel(ModelicaServices)")
omc_call(omc, "loadFile(\"$DYNAWO_PKG_PATH\")")

# 2. Load the selected model and validate the configuration against it
omc_call(omc, "loadFile(\"$MODEL_FILE_PATH\")")
check_user_configuration_single(omc;
    model = MODEL,
    slack_component = SLACK_COMPONENT,
    init_model_by_component = INIT_MODEL_BY_COMPONENT,
)

[ Info: Path to zmq file="/tmp/openmodelica.clarafercas.port.julia.S3v95shj3u"


Configuration checked successfully


In [3]:
# Cell 3: Auxiliary Model Setup

AUX_MODEL = MODEL * "_auxiliary"
AUX_FILE = joinpath(MODEL_DIR, AUX_MODEL * ".mo")


"/home/clarafercas/dynawo-notebooks/OpenModelica_only_users/BuildAux/models/MyBESS_auxiliary.mo"

In [4]:
# Cell 4: Single-Model Auxiliary Build Pipeline

# Create/refresh the auxiliary model in OpenModelica
sendExpression(omc, "deleteClass($AUX_MODEL)")
omc_call(omc, "copyClass($MODEL, \"$AUX_MODEL\")")

# Build component dictionary from the source model
components = get_all_components(omc, MODEL)

# Apply dictionary-driven replacements
apply_replacements!(omc, MODEL, AUX_MODEL, components, SLACK_COMPONENT)

# Delete connections to cleanup targets
delete_connections!(omc, AUX_MODEL, components)

# Delete cleanup-target components
delete_components!(omc, AUX_MODEL, components)

# Add INIT models for the source model
add_init_models!(omc, MODEL, AUX_MODEL, components, INIT_MODEL_BY_COMPONENT, SLACK_COMPONENT)

# Add load-flow modifiers
apply_LF_modifiers!(omc, MODEL, AUX_MODEL, components)

# Add initial equations for the source model
add_init_equations!(omc, MODEL, AUX_MODEL, components, INIT_MODEL_BY_COMPONENT, SLACK_COMPONENT)

# Clean up the auxiliary model equations in place
clean_aux_equations!(omc, AUX_MODEL, components, SLACK_COMPONENT)

# Validate the build
chk = sendExpression(omc, "checkModel($AUX_MODEL)", parsed=false)
println(chk)

# Save the auxiliary model
omc_call(omc, "saveModel(\"$AUX_FILE\", $AUX_MODEL)")

"Check of MyBESS_auxiliary completed successfully.
Class MyBESS_auxiliary has 61 equation(s) and 61 variable(s).
17 of these are trivial equation(s)."



true